In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

In [ ]:
test = pd.read_csv('test.csv')
train = pd.read_csv('train.csv')

In [ ]:
train.head()
train.isna().sum()
train.info()

In [ ]:
target = train['Churn']
cat_cols = [
    'Sex',
    'IsSeniorCitizen',
    'HasPartner',
    'HasChild',
    'HasPhoneService',
    'HasMultiplePhoneNumbers',
    'HasInternetService',
    'HasOnlineSecurityService',
    'HasOnlineBackup',
    'HasDeviceProtection',
    'HasTechSupportAccess',
    'HasOnlineTV',
    'HasMovieSubscription',
    'HasContractPhone',
    'IsBillingPaperless',
    'PaymentMethod'
]
num_cols = [
    'ClientPeriod',
    'MonthlySpending',
    'TotalSpent'
]

In [ ]:
for df in (train, test):
    df['TotalSpent'] = pd.to_numeric(df['TotalSpent'], errors='coerce')

train['TotalSpent'].fillna(train['TotalSpent'].median(), inplace=True)
test['TotalSpent'].fillna(train['TotalSpent'].median(), inplace=True)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].hist(train['ClientPeriod'])
axes[1].hist(train['MonthlySpending'])
axes[2].hist(train['TotalSpent'])
plt.show()


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 20))
for i, cat in enumerate(cat_cols):
    cnt = train[cat].value_counts()
    ax = axes[i // 4, i % 4]
    ax.bar(cnt.index, cnt.values)
    ax.set_title(cat)
    if i == 15:
        ax.tick_params(axis='x', rotation=30)
plt.show()
plt.tight_layout()




In [ ]:
target.value_counts()

Classes are imbalanced

In [129]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer

from catboost import CatBoostClassifier, Pool

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


In [ ]:
scaler = StandardScaler()
encoder = OneHotEncoder(sparse_output=False)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train.drop(columns=['Churn']), target, test_size=0.2, random_state=42)
X_submission = test

In [ ]:
encoder.fit(X_train[cat_cols])
X_train_cat = encoder.transform(X_train[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])
X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num = scaler.transform(X_test[num_cols])
X_train_scaled = np.hstack([X_train_cat, X_train_num])
X_test_scaled = np.hstack([X_test_cat, X_test_num])
X_submission_cat = encoder.transform(X_submission[cat_cols])
X_submission_num = scaler.transform(X_submission[num_cols])
X_submission_scaled = np.hstack([X_submission_cat, X_submission_num])

In [ ]:
pd.DataFrame(X_train_cat, columns=encoder.get_feature_names_out(cat_cols)).head()

In [ ]:
model = LogisticRegressionCV(cv=10, random_state=42, scoring='roc_auc', refit=True)
model.fit(X_train_scaled, y_train)


In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:,1]


print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

In [127]:
cat_model = CatBoostClassifier(iterations=100, learning_rate=0.05, random_state=42, depth=4, eval_metric='AUC')
cat_model.fit(X_train, y_train, cat_features=cat_cols)


0:	total: 7.36ms	remaining: 728ms
1:	total: 9.23ms	remaining: 452ms
2:	total: 11.5ms	remaining: 371ms
3:	total: 15.5ms	remaining: 371ms
4:	total: 17.1ms	remaining: 326ms
5:	total: 18.5ms	remaining: 290ms
6:	total: 19.7ms	remaining: 262ms
7:	total: 21.3ms	remaining: 245ms
8:	total: 22.6ms	remaining: 228ms
9:	total: 24ms	remaining: 216ms
10:	total: 25.6ms	remaining: 207ms
11:	total: 27.3ms	remaining: 200ms
12:	total: 29.3ms	remaining: 196ms
13:	total: 30.9ms	remaining: 190ms
14:	total: 32.4ms	remaining: 183ms
15:	total: 34.4ms	remaining: 180ms
16:	total: 37.7ms	remaining: 184ms
17:	total: 39.6ms	remaining: 180ms
18:	total: 41.5ms	remaining: 177ms
19:	total: 43.6ms	remaining: 174ms
20:	total: 45ms	remaining: 169ms
21:	total: 47.2ms	remaining: 167ms
22:	total: 48.9ms	remaining: 164ms
23:	total: 50.1ms	remaining: 159ms
24:	total: 51.5ms	remaining: 155ms
25:	total: 52.5ms	remaining: 149ms
26:	total: 53.8ms	remaining: 145ms
27:	total: 55.2ms	remaining: 142ms
28:	total: 56.5ms	remaining: 138ms

In [128]:
y_pred_cat = cat_model.predict(X_test)
y_prob_cat = cat_model.predict_proba(X_test)[:,1]

print(confusion_matrix(y_test, y_pred_cat))
print(classification_report(y_test, y_pred_cat))
print("ROC AUC:", roc_auc_score(y_test, y_prob_cat))

[[495  47]
 [103  95]]
              precision    recall  f1-score   support

           0       0.83      0.91      0.87       542
           1       0.67      0.48      0.56       198

    accuracy                           0.80       740
   macro avg       0.75      0.70      0.71       740
weighted avg       0.79      0.80      0.79       740

ROC AUC: 0.8447994707219799


In [ ]:
def generate_submission(model, X_test):

    y_prob = model.predict_proba(X_test)[:, 1]

    return pd.DataFrame({'ID': test['ID'], 'Churn': y_prob})



In [ ]:
submission = generate_submission(model, X_submission_scaled)
submission.to_csv('submission.csv', index=False)

In [124]:
submission = generate_submission(cat_model, X_submission)
submission.to_csv('submission.csv', index=False)

In [111]:
num_transformer = StandardScaler()
cat_transformer = OneHotEncoder(handle_unknown='ignore')

preprocess = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols),
    ]
)


pipe = make_pipeline(
    preprocess,
    LogisticRegression(max_iter=1000)
)

In [112]:
param_grid = {
    'logisticregression__C': [0.01, 0.1, 1, 10],
    'logisticregression__penalty': ['l2'],
    'logisticregression__solver': ['lbfgs'],   # or 'liblinear' if needed
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',    # or 'accuracy', 'f1', etc., depending on task
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
best_model = grid.best_estimator_

,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'logisticregression__C': [0.01, 0.1, ...], 'logisticregression__penalty': ['l2'], 'logisticregression__solver': ['lbfgs']}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [114]:
y_prob_grid = best_model.predict_proba(X_test)[:, 1]
y_pred_grid = (y_prob_grid > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred_grid))
print(classification_report(y_test, y_pred_grid))
print("ROC AUC:", roc_auc_score(y_test, y_prob_grid))

[[486  56]
 [ 96 102]]
              precision    recall  f1-score   support

           0       0.84      0.90      0.86       542
           1       0.65      0.52      0.57       198

    accuracy                           0.79       740
   macro avg       0.74      0.71      0.72       740
weighted avg       0.78      0.79      0.79       740

ROC AUC: 0.8405549964590555


In [115]:
submission = generate_submission(best_model, X_submission)
submission.to_csv('submission.csv', index=False)

In [130]:
grid_model_cat = CatBoostClassifier(
    random_seed=42,
    verbose=False,
    eval_metric='AUC'
)

grid = {
    'iterations': [100, 300, 500, 1000],
    'learning_rate': [0.03, 0.05, 0.1],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [1, 3, 5, 7]
}

grid_model_cat.grid_search(
    grid,
    Pool(X_train, y_train, cat_features=cat_cols),
    verbose=100
)


bestTest = 0.8258623538
bestIteration = 35

Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.
0:	loss: 0.8258624	best: 0.8258624 (0)	total: 383ms	remaining: 54.8s

bestTest = 0.8243906834
bestIteration = 86

Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.

bestTest = 0.8272964815
bestIteration = 22

Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.

bestTest = 0.8244732772
bestIteration = 70

Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.

bestTest = 0.82491628
bestIteration = 83

Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.

bestTest = 0.8246309561
bestIteration

{'params': {'depth': 4,
  'learning_rate': 0.05,
  'l2_leaf_reg': 5,
  'iterations': 1000},
 'cv_results': defaultdict(list,
             {'iterations': [0,
               1,
               2,
               3,
               4,
               5,
               6,
               7,
               8,
               9,
               10,
               11,
               12,
               13,
               14,
               15,
               16,
               17,
               18,
               19,
               20,
               21,
               22,
               23,
               24,
               25,
               26,
               27,
               28,
               29,
               30,
               31,
               32,
               33,
               34,
               35,
               36,
               37,
               38,
               39,
               40,
               41,
               42,
               43,
               44,
               4

In [131]:
submission = generate_submission(grid_model_cat, X_submission)
submission.to_csv('submission.csv', index=False)